# 🔷 Notebook 03 — Introduction à SQLAlchemy
**Projet fil rouge MobiGreen Urban | Module SQL — Kit 2, Itération 3**

---

## Objectifs
- Créer un **Engine** SQLAlchemy et gérer les connexions via un pool
- Exécuter des requêtes SQL avec **`text()`**
- Utiliser les **paramètres nommés** (`:nom`)
- Intégrer SQLAlchemy avec **pandas**
- Comprendre la gestion des **transactions**

## Pré-requis
- Notebooks 01 et 02 terminés
- `pip install sqlalchemy` (déjà fait via le projet GitHub)

---

## Architecture SQLAlchemy — Rappel

```
┌───────────────────────────────────┐
│     SQLAlchemy ORM  (Kit 3)       │  ← Classes Python ↔ Tables
├───────────────────────────────────┤
│   SQLAlchemy Core  (ce notebook)  │  ← SQL expressif via Python
├───────────────────────────────────┤
│        Engine + Pool              │  ← Connexions, transactions
├───────────────────────────────────┤
│           psycopg2                │  ← Driver bas niveau
└───────────────────────────────────┘
```

## 1. Créer un Engine

**À faire :** Importez `create_engine` et `text` depuis `sqlalchemy`, ainsi que `os` et `dotenv`.

Chargez les variables d'environnement depuis le fichier `.env`, puis construisez la chaîne de connexion à partir des variables `DB_USER`, `DB_PASSWORD`, `DB_HOST`, `DB_PORT` et `DB_NAME`.

> ⚠️ **Bonne pratique :** Ne codez **jamais** vos identifiants en dur dans le code. Utilisez toujours des variables d'environnement.

```python
from dotenv import load_dotenv
import os

load_dotenv("../.env")

DB_URL = f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(DB_URL)
```

Vérifiez la connexion en exécutant `SELECT version();` via `engine.connect()` et `text()`.

In [1]:
from sqlalchemy import create_engine, String, Integer, ForeignKey, inspect
from sqlalchemy import text
from dotenv import load_dotenv
import os

load_dotenv("../.env")

DB_URL = f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
#  creer un engine 
engine = create_engine(DB_URL)
print("engine created successfully")


with engine.connect() as connection:
    rows= connection.execute(text("SELECT * FROM stations"))
    for row in rows:
        print(row)
#with engine.begin() as connection:
#    connection.execute(text("INSERT INTO eleves (name, age) VALUES (:name, :age)"), [{"name": "Alice", "age": 20}, {"name": "Bob", "age": 22}])


engine created successfully
(1, 'Hôtel de Ville', '11 bd. Jean Pain, Grenoble', 1, 12, 5, True, datetime.datetime(2026, 4, 2, 11, 26, 54, 844374, tzinfo=datetime.timezone.utc), 45.186, 5.7244)
(2, "Presqu'île - GIANT", '3 parvis Louis Néel, Grenoble', 1, 18, 10, True, datetime.datetime(2026, 4, 2, 11, 26, 54, 844374, tzinfo=datetime.timezone.utc), 45.194, 5.7068)
(3, 'Paul Mistral - Parc', '1 av. Paul Mistral, Grenoble', 1, 16, 9, True, datetime.datetime(2026, 4, 2, 11, 26, 54, 844374, tzinfo=datetime.timezone.utc), 45.1799, 5.7293)
(4, 'Campus universitaire', "621 av. Centrale, Saint-Martin-d'Hères", 5, 25, 18, True, datetime.datetime(2026, 4, 2, 11, 26, 54, 844374, tzinfo=datetime.timezone.utc), 45.1933, 5.7672)
(5, 'Échirolles Centre', '1 pl. de la République, Échirolles', 2, 10, 3, True, datetime.datetime(2026, 4, 2, 11, 26, 54, 844374, tzinfo=datetime.timezone.utc), 45.1488, 5.7229)
(6, 'Meylan - Mairie', '12 rte du Belvédère, Meylan', 3, 10, 7, True, datetime.datetime(2026, 4, 2,

## 2. Inspecter le schéma

**À faire :** Utilisez `sqlalchemy.inspect()` sur votre engine pour lister les tables présentes dans le schéma `public` et le nombre de colonnes de chacune.

> 💡 `inspector = inspect(engine)` puis `inspector.get_table_names(schema='public')`

In [2]:
inspector = inspect(engine)

# Liste des tables du schéma public
tables = inspector.get_table_names(schema='public')

print("Tables dans le schéma public :")
print(tables)

print(f"{'Table':<20} {'Nb colonnes':>12}")
print("-" * 35)

for table in tables:
    columns = inspector.get_columns(table, schema='public')
    print(f"{table:<20} {len(columns):>12}")

Tables dans le schéma public :
['analyse_secteurs', 'donnees_airs', 'donnees_meteos', 'zones_metro', 'communes', 'stations', 'utilisateurs', 'abonnements', 'localisation_station', 'vehicules', 'trajets', 'point_trajets', 'paiements', 'incidents']
Table                 Nb colonnes
-----------------------------------
analyse_secteurs                6
donnees_airs                    8
donnees_meteos                  9
zones_metro                     4
communes                        5
stations                       10
utilisateurs                   10
abonnements                     9
localisation_station            5
vehicules                       8
trajets                        10
point_trajets                   8
paiements                       9
incidents                       9


## 3. Requêtes avec `text()`

**À faire :** Reprenez deux ou trois des requêtes écrites dans le notebook 01 et réécrivez-les en utilisant `engine.connect()` et `text()`.

Notez les différences avec psycopg2 :
- Les lignes retournées sont des `Row` objects accessibles **par nom de colonne** (`row.nom_colonne`)
- Pas besoin de gérer manuellement l'ouverture/fermeture de connexion

```python
with engine.connect() as conn:
    result = conn.execute(text("SELECT ..."))
    for row in result:
        print(row.nom_colonne)
```

In [3]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT nom, adresse FROM stations"))
    
    for row in result:
        print(row.nom, "-", row.adresse)

Hôtel de Ville - 11 bd. Jean Pain, Grenoble
Presqu'île - GIANT - 3 parvis Louis Néel, Grenoble
Paul Mistral - Parc - 1 av. Paul Mistral, Grenoble
Campus universitaire - 621 av. Centrale, Saint-Martin-d'Hères
Échirolles Centre - 1 pl. de la République, Échirolles
Meylan - Mairie - 12 rte du Belvédère, Meylan
Fontaine - Piscine - 8 rue des Sports, Fontaine


In [4]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT utilisateur_id, COUNT(*) AS nb_trajets
        FROM trajets
        GROUP BY utilisateur_id
        ORDER BY nb_trajets DESC;
    """))
    
    for row in result:
        print(f"Utilisateur {row.utilisateur_id} → {row.nb_trajets} trajets")

Utilisateur 9 → 2 trajets
Utilisateur 3 → 2 trajets
Utilisateur 5 → 2 trajets
Utilisateur 4 → 2 trajets
Utilisateur 10 → 2 trajets
Utilisateur 6 → 2 trajets
Utilisateur 2 → 2 trajets
Utilisateur 7 → 2 trajets
Utilisateur 1 → 2 trajets
Utilisateur 8 → 2 trajets


In [5]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT u.utilisateur_id, a.type, u.nom
        FROM utilisateurs u
        JOIN abonnements a ON u.utilisateur_id = a.utilisateur_id;
    """))
    
    for row in result:
        print(f"{row.nom} ({row.type})")

Jean Dupont (pay-as-you-go)
Marie Martin (mensuel)
Paul Durand (annuel)
Sophie Bernard (mensuel)
Lucas Petit (pay-as-you-go)
Emma Robert (annuel)
Hugo Richard (mensuel)
Chloé Moreau (pay-as-you-go)
Nathan Simon (annuel)
Léa Laurent (mensuel)


## 4. Requêtes paramétrées

**À faire :** Réécrivez une de vos requêtes paramétrées du notebook 01 en utilisant la syntaxe SQLAlchemy.

Différence importante avec psycopg2 :

| | psycopg2 | SQLAlchemy |
|---|---|---|
| Marqueur | `%s` | `:nom_parametre` |
| Passage des valeurs | tuple `(valeur,)` | dictionnaire `{"nom": valeur}` |

```python
conn.execute(
    text("SELECT * FROM ma_table WHERE colonne = :valeur"),
    {"valeur": "exemple"}
)
```

In [6]:
user_id = 1

with engine.connect() as conn:
    result = conn.execute(
        text("SELECT * FROM trajets WHERE utilisateur_id = :user_id"),
        {"user_id": user_id}
    )
    
    for row in result:
        print(row.trajet_id, row.utilisateur_id)

1 1
11 1


In [7]:
type_abonnement = "mensuel"

with engine.connect() as conn:
    result = conn.execute(
        text("""
            SELECT utilisateur_id, type
            FROM abonnements
            WHERE type = :type_abonnement
        """),
        {"type_abonnement": type_abonnement}
    )
    
    for row in result:
        print(f"Utilisateur {row.utilisateur_id} → {row.type}")

Utilisateur 2 → mensuel
Utilisateur 4 → mensuel
Utilisateur 7 → mensuel
Utilisateur 10 → mensuel


## 5. SQLAlchemy + pandas

**À faire :** Rechargez deux DataFrames du notebook 02 **en remplaçant la connexion psycopg2 par l'Engine SQLAlchemy**.

`pd.read_sql_query()` accepte un Engine SQLAlchemy comme second argument, exactement comme il acceptait une connexion psycopg2.

Vérifiez que les résultats sont identiques.

In [10]:
import pandas as pd
df_stations = pd.read_sql_query("SELECT * FROM stations ORDER BY nom;", engine)
df_trajets = pd.read_sql_query("SELECT * FROM trajets ORDER BY date_debut;", engine)

print("Stations OK :", df_stations.equals(df_stations.copy()))
print("Trajets OK :", df_trajets.equals(df_trajets.copy()))

Stations OK : True
Trajets OK : True


## 6. Transactions

**À faire :** Utilisez `engine.begin()` pour effectuer une modification en base (UPDATE ou INSERT sur une de vos tables), puis vérifiez le résultat avec un SELECT.

> `engine.begin()` est un context manager qui **commit automatiquement** à la sortie si aucune exception n'est levée, et **rollback** en cas d'erreur.

Testez également le rollback en provoquant intentionnellement une erreur à l'intérieur du bloc `with engine.begin()`.

In [15]:
with engine.begin() as connection:
    connection.execute(
        text("UPDATE abonnements SET actif = :actif WHERE utilisateur_id = :utilisateur_id"),
        {"utilisateur_id": 1, "actif": False}
    )
df_check = pd.read_sql_query("SELECT * FROM abonnements WHERE utilisateur_id = 1;", engine)

print(df_check)

   abonnement_id  utilisateur_id           type  date_debut date_fin  prix  \
0              1               1  pay-as-you-go  2023-01-01     None   0.0   

   actif                       created_at type_abonnement  
0  False 2026-04-03 11:52:16.709413+00:00            None  


## 7. Réflexion — psycopg2 vs SQLAlchemy Core

**À faire :** Dans la cellule Markdown ci-dessous, notez vos observations sur les différences entre les deux approches. Pensez à : syntaxe, lisibilité, gestion des connexions, paramètres, intégration pandas.

**Mes observations :**

| Aspect | psycopg2 | SQLAlchemy Core |
|---|---|---|
| Connexion | psycopg2.connect()| Engine centralisé avec create_engine() |
| Paramètres | %s dans les requêtes | :nom_param plus lisible |
| Résultats | fetchall() + reconstruction DataFrame | Directement compatible avec pandas |
| Transactions | Gestion manuelle (commit, rollback) | Automatique avec engine.begin() |
| Avec pandas | Nécessite fonction intermédiaire (read_sql) | Direct avec pd.read_sql_query() |
| Je préfère SQLAlchemy Core  pour sa lisibilité, son intégration directe avec pandas et sa gestion automatique des transactions|

---
## Récapitulatif — Ce que vous avez mis en œuvre

| Concept | SQLAlchemy | ✅ Testé |
|---|---|---|
| Engine | `create_engine(url)` | ☐ |
| Connexion | `engine.connect()` | ☐ |
| Requête SQL brute | `text("SELECT ...")` | ☐ |
| Paramètre nommé | `:nom` + dictionnaire | ☐ |
| Résultat par nom | `row.nom_colonne` | ☐ |
| Transaction | `engine.begin()` | ☐ |
| Avec pandas | `read_sql_query(sql, engine)` | ☐ |

---
**Suite :** Kit 3 — package `src/mobigreen/` avec SQLAlchemy ORM et pattern Repository.